# 01 · EDA: data shape, stores and promotions

First look at the Newcastle sausage roll data. Three questions:

1. **Shape** - what do we have, and what is wrong with it?
2. **Stores** - how do the three stores differ on a normal day?
3. **Promotions** - what ran, where and when, and how do sales look on those days? (`promotion` and `other_promotion`)

Nothing here is a conclusion about which promotion works best. The raw comparisons in section 3 are
deliberately naive; the baseline notebook (`02_`) does the fair comparison.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
    "axes.titlelocation": "left", "axes.titlesize": 11, "axes.titleweight": "bold",
})
BLUE, GREY = "#2a78d6", "#b9b8b2"
STORE_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a"]  # one fixed colour per store

raw = pd.read_excel("../data/newcastle_sausage_roll_pricing_challenge_20261022.xlsx")

## 1. Data shape

In [3]:
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"{raw.date.min():%Y-%m-%d} to {raw.date.max():%Y-%m-%d}: {raw.date.nunique()} days x {raw.store_id.nunique()} stores = {raw.date.nunique() * raw.store_id.nunique():,} expected rows")
raw.head()

2,192 rows x 17 columns
2024-07-20 to 2026-07-19: 730 days x 3 stores = 2,190 expected rows


,date,store_id,location,store_description,day_of_week,temperature_c,rain,rain_mm,home_match_day,school_holidays,promotion,other_promotion,total_shop_transactions,sausage_roll_transactions,sausage_rolls_sold,sausage_roll_takings_gbp,total_shop_takings_gbp
0,2024-07-20,NCL-KP-03,Kingston Park,Retail-park branch with parking and family-led...,Saturday,17.50,1,7.00,0,1,none,none,411,88,0,127.20,"2,199.50"
1,2024-07-20,NCL-OQ-01,Quayside,Office-quarter branch serving weekday commuter...,Saturday,17.50,1,7.00,0,1,none,none,175,30,36,43.20,793.69
2,2024-07-20,NCL-SJ-02,St James' Quarter,"City-centre branch near shopping, nightlife, a...",Saturday,17.50,1,7.00,0,1,none,none,444,73,96,115.20,"2,034.30"
3,2024-07-21,NCL-KP-03,Kingston Park,Retail-park branch with parking and family-led...,Sunday,15.80,0,0.00,0,1,none,none,349,61,75,90.00,"1,603.50"
4,2024-07-21,NCL-OQ-01,Quayside,Office-quarter branch serving weekday commuter...,Sunday,15.80,0,0.00,0,1,none,none,162,28,34,40.80,819.88


In [4]:
pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "unique": raw.nunique()})

,dtype,missing,unique
date,datetime64[us],0,730
store_id,str,0,3
location,str,0,3
store_description,str,0,3
day_of_week,str,0,7
temperature_c,float64,0,202
rain,int64,0,2
rain_mm,float64,0,111
home_match_day,int64,0,2
school_holidays,int64,0,2


### Data quality checks

Each line counts **rows** with the problem. Anything non-zero gets fixed (or blanked) in the next cell.

In [5]:
count_cols = ["total_shop_transactions", "sausage_roll_transactions", "sausage_rolls_sold"]
checks = pd.Series({
    "duplicate rows": raw.duplicated().sum(),
    "missing takings": raw.sausage_roll_takings_gbp.isna().sum(),
    "999999 placeholder value": (raw.select_dtypes("number") == 999999).any(axis=1).sum(),
    "negative counts": (raw[count_cols] < 0).any(axis=1).sum(),
    "negative rain_mm": (raw.rain_mm < 0).sum(),
    "temperature above 40C": (raw.temperature_c > 40).sum(),
    "rolls sold < roll transactions": (raw.sausage_rolls_sold < raw.sausage_roll_transactions).sum(),
    "rain flag disagrees with rain_mm": (raw.rain != (raw.rain_mm > 0)).sum(),
    "day_of_week disagrees with date": (raw.date.dt.day_name() != raw.day_of_week).sum(),
}, name="rows")
checks.to_frame()

,rows
duplicate rows,2
missing takings,3
999999 placeholder value,3
negative counts,3
negative rain_mm,3
temperature above 40C,1
rolls sold < roll transactions,2
rain flag disagrees with rain_mm,0
day_of_week disagrees with date,0


Cleaning rule: **drop exact duplicates, blank out impossible values** (set to `NaN`) rather than guess a fix.
A row with one bad cell keeps its other columns. `NaN`s are ignored by `mean()`.

In [6]:
df = raw.drop_duplicates().replace(999999, np.nan)
df[count_cols] = df[count_cols].mask(df[count_cols] < 0)
df["rain_mm"] = df.rain_mm.mask(df.rain_mm < 0)
df["temperature_c"] = df.temperature_c.mask(df.temperature_c > 40)
df["sausage_rolls_sold"] = df.sausage_rolls_sold.mask(df.sausage_rolls_sold < df.sausage_roll_transactions)

# a few simple derived columns used below
df["avg_price_gbp"] = df.sausage_roll_takings_gbp / df.sausage_rolls_sold
df["rolls_per_transaction"] = df.sausage_rolls_sold / df.sausage_roll_transactions
df["roll_share_of_transactions"] = df.sausage_roll_transactions / df.total_shop_transactions
df["month"] = df.date.dt.to_period("M").dt.to_timestamp()
df["offer"] = df.promotion.where(df.promotion != "none", df.other_promotion)  # promotion and other_promotion never overlap (section 3)

print(f"{len(raw) - len(df)} duplicate rows dropped; {len(df):,} rows left")
df.describe().T

2 duplicate rows dropped; 2,190 rows left


,count,mean,min,25%,50%,75%,max,std
date,2190,2025-07-19 12:00:00,2024-07-20 00:00:00,2025-01-18 00:00:00,2025-07-19 12:00:00,2026-01-18 00:00:00,2026-07-19 00:00:00,NaN
temperature_c,"2,189.00",9.53,-3.20,4.50,9.90,14.30,20.00,5.59
rain,"2,190.00",0.40,0.00,0.00,0.00,1.00,1.00,0.49
rain_mm,"2,187.00",1.86,0.00,0.00,0.00,2.60,26.80,3.39
home_match_day,"2,190.00",0.08,0.00,0.00,0.00,0.00,1.00,0.27
school_holidays,"2,190.00",0.25,0.00,0.00,0.00,1.00,1.00,0.44
total_shop_transactions,"2,189.00",320.27,107.00,281.00,317.00,360.00,793.00,78.86
sausage_roll_transactions,"2,189.00",60.12,16.00,49.00,58.00,69.00,271.00,18.12
sausage_rolls_sold,"2,185.00",79.37,18.00,64.00,76.00,91.00,380.00,26.72
sausage_roll_takings_gbp,"2,184.00",101.61,23.40,82.50,97.50,117.45,380.00,31.56


Sanity check after cleaning: the average price per roll (takings / rolls sold) on days with no offer
should be the list price. Is it constant?

In [7]:
list_price = df[df.offer == "none"].groupby(df.date.dt.to_period("Q")).avg_price_gbp.agg(["min", "mean", "max", "count"])
list_price

,min,mean,max,count
date,,,,
2024Q3,1.20,1.24,1.25,203
2024Q4,1.25,1.25,1.25,207
2025Q1,1.25,1.29,1.30,206
2025Q2,1.30,1.30,1.30,209
2025Q3,1.30,1.30,1.30,260
2025Q4,1.30,1.30,1.30,206
2026Q1,1.30,1.34,1.35,193
2026Q2,1.35,1.35,1.35,263
2026Q3,1.35,1.35,1.35,57


The list price is **not constant**: it steps up over time. Promotions that ran at different times therefore started from
different base prices, which matters when we compare them.

## 2. Stores

Three stores, each with a distinct customer mix (from `store_description`).
To see how stores differ naturally, we use **days with no promotion of any kind**.

In [ ]:
stores = df.groupby(["store_id", "location"]).agg(days=("date", "count"), description=("store_description", "first")).reset_index()
stores.style.set_properties(subset=["description"], **{"white-space": "normal", "text-align": "left"}).hide(axis="index")

In [ ]:
normal = df[df.offer == "none"]

store_summary = normal.groupby("location").agg(
    days=("date", "count"),
    shop_transactions=("total_shop_transactions", "mean"),
    roll_transactions=("sausage_roll_transactions", "mean"),
    rolls_sold=("sausage_rolls_sold", "mean"),
    roll_takings_gbp=("sausage_roll_takings_gbp", "mean"),
    shop_takings_gbp=("total_shop_takings_gbp", "mean"),
    roll_share_of_transactions=("roll_share_of_transactions", "mean"),
    avg_price_gbp=("avg_price_gbp", "mean"),
)
store_summary

In [ ]:
order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = normal.assign(dow=normal.date.dt.day_of_week).groupby(["dow", "location"]).sausage_rolls_sold.mean().unstack()
monthly = normal.groupby(["month", "location"]).sausage_rolls_sold.mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
for ax, data, title in [(axes[0], dow, "By day of week"), (axes[1], monthly, "By month")]:
    for col, colour in zip(data.columns, STORE_COLOURS):
        ax.plot(data.index, data[col], color=colour, lw=2, label=col)
    ax.set_title(title)
axes[0].set_xticks(range(7), order)
axes[0].set_ylabel("Avg rolls sold per day (no-promo days)")
axes[1].legend(frameon=False, loc="upper left", ncol=3)
plt.tight_layout()

In [ ]:
# do school holidays, home matches and rain move sales, and does it differ by store?
context = pd.concat({
    "school holiday": normal.groupby(["location", "school_holidays"]).sausage_rolls_sold.mean().unstack().rename(columns={0: "no", 1: "yes"}),
    "home match": normal.groupby(["location", "home_match_day"]).sausage_rolls_sold.mean().unstack().rename(columns={0: "no", 1: "yes"}),
    "rain": normal.groupby(["location", "rain"]).sausage_rolls_sold.mean().unstack().rename(columns={0: "no", 1: "yes"}),
}, axis=1)
context

## 3. Promotions

Two columns: `promotion` (the sausage roll promotion we are choosing between) and `other_promotion`
(shop-wide offers on other products that might also move sausage roll sales).

### 3.1 What ran

In [ ]:
pd.concat({
    "promotion": df.promotion.value_counts(),
    "other_promotion": df.other_promotion.value_counts(),
}, axis=1).fillna(0).astype(int)

In [ ]:
# do the two ever run at the same time?
pd.crosstab(df.promotion, df.other_promotion)

### 3.2 When and where they ran

Collapse consecutive promo days at a store into a single "run" (start, end, length).

In [ ]:
def runs(col):
    d = df[df[col] != "none"].sort_values(["store_id", "date"])
    new_run = (d.store_id != d.store_id.shift()) | (d[col] != d[col].shift()) | (d.date.diff().dt.days != 1)
    return d.groupby(new_run.cumsum()).agg(
        store=("location", "first"), offer=(col, "first"), start=("date", "min"), end=("date", "max"), days=("date", "count"),
    ).reset_index(drop=True)

promo_runs, other_runs = runs("promotion"), runs("other_promotion")
promo_runs.assign(start=promo_runs.start.dt.date, end=promo_runs.end.dt.date).sort_values("start")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5.2), sharex=True)
for ax, r, title in [(axes[0], promo_runs, "promotion"), (axes[1], other_runs, "other_promotion")]:
    locs = sorted(df.location.unique())[::-1]
    r = r.sort_values("start").assign(n=lambda x: x.groupby("store").cumcount())
    for _, row in r.iterrows():
        y = locs.index(row.store)
        ax.barh(y, (row.end - row.start).days + 1, left=row.start, height=0.4, color=GREY, edgecolor="#6b6a65")
        # alternate label height so neighbouring labels don't collide
        ax.text(row.start + (row.end - row.start) / 2, y + 0.25 + 0.2 * (row.n % 2), row.offer.replace("_", " "), ha="center", va="bottom", fontsize=7)
    ax.set_xlim(df.date.min(), df.date.max())
    ax.set_yticks(range(len(locs)), locs)
    ax.set_ylim(-0.5, len(locs) - 0.1)
    ax.set_title(title)
plt.tight_layout()

In [ ]:
# is the same promotion ever running in two stores on the same day?
print("max stores sharing a promotion on one day:", df[df.promotion != "none"].groupby("date").store_id.nunique().max())
print("max stores sharing an other_promotion on one day:", df[df.other_promotion != "none"].groupby(["date", "other_promotion"]).store_id.nunique().max())

### 3.3 Raw sales by offer

Average per store-day. `none` = no promotion and no other promotion.
**Naive on purpose:** it ignores which store, season and day of week each promo ran on.

In [ ]:
by_offer = df.groupby("offer").agg(
    days=("date", "count"),
    rolls_sold=("sausage_rolls_sold", "mean"),
    roll_transactions=("sausage_roll_transactions", "mean"),
    rolls_per_transaction=("rolls_per_transaction", "mean"),
    avg_price_gbp=("avg_price_gbp", "mean"),
    roll_takings_gbp=("sausage_roll_takings_gbp", "mean"),
    shop_transactions=("total_shop_transactions", "mean"),
    shop_takings_gbp=("total_shop_takings_gbp", "mean"),
).sort_values("rolls_sold", ascending=False)
by_offer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, col, title in [(axes[0], "rolls_sold", "Rolls sold per store-day"), (axes[1], "roll_takings_gbp", "Roll takings per store-day (£)")]:
    d = by_offer[col].sort_values()
    bars = ax.barh(d.index, d.values, color=[GREY if o == "none" else BLUE for o in d.index], height=0.6)
    ax.bar_label(bars, fmt="%.0f", padding=3, fontsize=8)
    ax.set_title(title)
    ax.grid(axis="y", visible=False)
plt.tight_layout()

### 3.4 Is the comparison fair?

If promotions landed on unusual days (school holidays, home matches, a particular season), the raw gap above
partly reflects that and not the promotion. Compare the *mix of days* each offer ran on.

In [ ]:
mix = df.assign(weekend=df.date.dt.day_of_week >= 5).groupby("offer").agg(
    school_holidays=("school_holidays", "mean"),
    home_match_day=("home_match_day", "mean"),
    weekend=("weekend", "mean"),
    rain=("rain", "mean"),
    temperature_c=("temperature_c", "mean"),
    first_day=("date", "min"),
    last_day=("date", "max"),
)
mix.assign(first_day=mix.first_day.dt.date, last_day=mix.last_day.dt.date)

In [ ]:
# same comparison split by store: is the pattern consistent across stores?
df.pivot_table(index="offer", columns="location", values="sausage_rolls_sold", aggfunc="mean").loc[by_offer.index]

## What I found

**Shape and quality**
- 2,192 rows x 17 columns: 730 days x 3 stores = 2,190 expected, plus 2 duplicate rows (dropped).
- A handful of impossible values, all blanked rather than guessed: three `999999` placeholders, three negative counts, three negative `rain_mm`, one 125°C temperature, two rows where rolls sold < roll transactions, three missing takings.
- The list price is not constant: it steps up from about £1.20 (mid-2024) to £1.25, £1.30 and £1.35 (2026).

**Stores**
- Quayside is the office-quarter store: steady ~70 rolls on weekdays, collapsing to ~40 at weekends, and lower in school holidays.
- St James' Quarter peaks on Fridays and Saturdays, and sells far more on home-match days (~133 vs ~81 rolls).
- Kingston Park is the weekend / school-holiday store.
- Stores differ in level as well as shape, so any promotion comparison must be made within a store.

**Promotions**
- Three roll promotions: `20_percent_off` (2 weeks per store, autumn 2024), `two_for_2` (2 weeks per store, autumn 2025), `one_pound_week` (1 week per store, spring 2026).
- Each one ran in **one store at a time, a few weeks apart**, so the other two stores trade normally at the same time. That staggering is our best tool for a fair comparison.
- `other_promotion` offers (drinks, breakfast, etc.) never overlap with a roll promotion and run in **all three stores at once**, so there is no same-day control for them.
- Raw averages: `one_pound_week` (155 rolls/day) and `two_for_2` (147) sell about twice the no-offer day (76); `20_percent_off` sells 102. In takings, `20_percent_off` is only ~£102 vs £99 on a normal day, while the other two are ~£155.

**Caution (the raw comparison is not yet fair)**
- Promotions ran in different seasons, at different list prices and on different mixes of school-holiday and home-match days (e.g. `one_pound_week`: 33% school holidays, 14% home matches).
- Next step (`02_baseline`): compare each promo run with the same store's non-promo days at the same time, and the other two stores on the same days.